## Task 2: Logging and visualizing training progress

#### Goal: Use TensorBoard and SummaryWriter to log and visualize the loss function on the training and validation sets.

1. Reuse a model and dataset from a previous lab (e.g. the regression or classification MLP from Lab 4).
2. Create a `SummaryWriter` and log both training loss and validation loss at each epoch using `writer.add_scalars()`, storing them under the same `main_tag` (e.g. `'loss'`) with keys `'training'` and `'validation'`.
3. Call `writer.flush()` at the end of each epoch.
4. Open TensorBoard and inspect the loss curves for both splits.

**Assignment:** Implement loss function logging for both training and validation sets using SummaryWriter. Visualize the training progress in TensorBoard and verify that the curves behave as expected (e.g. decreasing loss, no divergence).

In [22]:
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [23]:
from torch.utils.tensorboard import SummaryWriter

deepWriter = SummaryWriter("runs/task_2/deep")
wideWriter = SummaryWriter("runs/task_2/wide")
pyramidWriter = SummaryWriter("runs/task_2/pyramid")

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
from ucimlrepo import fetch_ucirepo

# Dry Bean Dataset - 13611 samples, 16 features, 7 classes
dry_bean = fetch_ucirepo(id=602)

X = dry_bean.data.features.values.astype("float32")
y = LabelEncoder().fit_transform(dry_bean.data.targets.values.ravel())

print(f"Dataset:  Dry Bean")
print(f"Samples:  {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes:  {np.unique(dry_bean.data.targets.values).tolist()}")
print(f"Class counts: {np.bincount(y)}")

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval)

scaler = StandardScaler().fit(X_trainval)
X_train = scaler.transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")
X_val = scaler.transform(X_val).astype("float32")

input_dim = X_trainval.shape[1]
n_classes = len(np.unique(y))
output_dim = n_classes

print(f"\nX_trainval: {X_trainval.shape}")
print(f"X_test:     {X_test.shape}")
print(f"n_classes:  {n_classes}")

Dataset:  Dry Bean
Samples:  13611
Features: 16
Classes:  ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']
Class counts: [1322  522 1630 3546 1928 2027 2636]

X_trainval: (10888, 16)
X_test:     (2723, 16)
n_classes:  7


In [21]:
from utils import wideMLP, deepMLP, pyramidMLP, train_clf
from torch.utils.tensorboard import SummaryWriter

configs = [
    ("wide", wideMLP()),
    ("deep", deepMLP()),
    ("pyramid", pyramidMLP()),
]

checkpoints = {}
for name, model in configs:
    writer = SummaryWriter(f"runs/task_2/{name}")
    checkpoints[name] = train_clf(model, X_train, y_train, X_val, y_val, writer)
    print(f"  best val loss: {checkpoints[name]['best_val_loss']:.4f} "
          f"| best val acc: {checkpoints[name]['best_val_acc']:.4f}")

  best val loss: 0.1749 | best val acc: 0.9357
  best val loss: 0.2592 | best val acc: 0.9197
  best val loss: 0.1876 | best val acc: 0.9348
